In [ ]:
import numpy as np
from windcalc.windengine import BuildingWindEngine, SurfaceType
from typing import List, Dict, Tuple, Optional

def close_polygon(polygon: np.ndarray, tol: float = 1e-8) -> np.ndarray:
    """Poligonu kapatır (ilk nokta = son nokta)."""
    polygon = np.asarray(polygon, dtype=float)
    if len(polygon) == 0:
        return polygon
    if not np.allclose(polygon[0], polygon[-1], atol=tol):
        polygon = np.vstack([polygon, polygon[0]])
    return polygon


def _open_polygon(polygon: np.ndarray, tol: float = 1e-8) -> np.ndarray:
    """Kapalı poligondan tekrar eden son noktayı kaldırır."""
    polygon = np.asarray(polygon, dtype=float)
    if len(polygon) > 1 and np.allclose(polygon[0], polygon[-1], atol=tol):
        return polygon[:-1]
    return polygon


def polygon_edges(polygon: np.ndarray) -> List[Dict]:
    """Poligonun kenarlarını (p1, p2, index) listesi olarak döndürür."""
    pts = _open_polygon(polygon)
    n = len(pts)
    return [
        {"index": i, "p1": pts[i], "p2": pts[(i + 1) % n]}
        for i in range(n)
    ]

def polygon_edges_to_render(polygon: np.ndarray) -> List[Dict]:
    """Poligonun kenarlarını (p1, p2, index) listesi olarak döndürür."""
    pts = close_polygon(polygon)
    n = len(pts)
    return [
        [pts[i], pts[(i + 1) % n]]
        for i in range(n)
    ]


def _compute_normal(polygon: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Newell metodu ile poligon normalini hesaplar."""
    pts = _open_polygon(polygon)
    if len(pts) < 3:
        raise ValueError("Poligon en az 3 noktadan oluşmalıdır.")

    # Vektörel Newell formülü
    p_curr = pts
    p_next = np.roll(pts, -1, axis=0)
    normal = np.array([
        np.sum((p_curr[:, 1] - p_next[:, 1]) * (p_curr[:, 2] + p_next[:, 2])),
        np.sum((p_curr[:, 2] - p_next[:, 2]) * (p_curr[:, 0] + p_next[:, 0])),
        np.sum((p_curr[:, 0] - p_next[:, 0]) * (p_curr[:, 1] + p_next[:, 1])),
    ])

    norm_len = np.linalg.norm(normal)
    if norm_len < 1e-12:
        raise ValueError(f"Geçersiz/çökmüş poligon düzlemi: {pts}")
    return normal, normal / norm_len


def _get_best_projection_plane(polygon: np.ndarray) -> Tuple[Tuple[int, int], int]:
    """En iyi projeksiyon düzlemini seçer. ((i1,i2), missing_idx) döndürür."""
    _, normal_unit = _compute_normal(polygon)
    missing_idx = int(np.argmax(np.abs(normal_unit)))
    axes = [(0, 1), (0, 2), (1, 2)]
    # missing_idx 2 -> XY, 1 -> XZ, 0 -> YZ
    plane_map = {2: (0, 1), 1: (0, 2), 0: (1, 2)}
    return plane_map[missing_idx], missing_idx


def point_unproject_to_3d(polygon: np.ndarray, pt_2d: np.ndarray) -> np.ndarray:
    """2D noktayı poligon düzlemine geri yansıtır."""
    normal_unit = _compute_normal(polygon)[1]
    (i1, i2), missing_idx = _get_best_projection_plane(polygon)

    n_missing = normal_unit[missing_idx]
    D = -np.dot(normal_unit, polygon[0])

    p3d = np.zeros(3)
    p3d[i1] = pt_2d[0]
    p3d[i2] = pt_2d[1]
    p3d[missing_idx] = -(normal_unit[i1] * pt_2d[0]
                         + normal_unit[i2] * pt_2d[1] + D) / n_missing
    return p3d

def _clip_line_to_polygon(line_p1, line_p2, polygon, i1, i2):
    """Sonsuz çizginin poligon sınırıyla kesiştiği iki noktayı bulur."""

    p1 = np.asarray(line_p1, dtype=float)
    p2 = np.asarray(line_p2, dtype=float)

    direction = p2 - p1
    intersections = []

    for i in range(len(polygon)):
        a = np.asarray(polygon[i], dtype=float)[[i1, i2]]
        b = np.asarray(polygon[(i + 1) % len(polygon)], dtype=float)[[i1, i2]]

        edge = b - a

        cross = direction[0] * edge[1] - direction[1] * edge[0]

        if abs(cross) < 1e-10:
            continue

        q = a - p1

        t = (q[0] * edge[1] - q[1] * edge[0]) / cross
        u = (q[0] * direction[1] - q[1] * direction[0]) / cross

        # Poligon kenarı üzerinde kesişim
        if -1e-10 <= u <= 1 + 1e-10:
            intersections.append(p1 + t * direction)

    if len(intersections) < 2:
        return None

    # Çizgi üzerindeki en uzak iki kesişim
    intersections.sort(
        key=lambda p: np.dot(p - p1, direction)
    )

    return intersections[0], intersections[-1]

def offset_edge(polygon, p1_3d, p2_3d, w_vector, e, perp=False):
    """Kenarı w_vector yönünde veya ona dik yönde offsetler."""
    (i1, i2), _ = _get_best_projection_plane(polygon)

    EXT = 100

    p1 = np.asarray(p1_3d, dtype=float)[[i1, i2]]
    p2 = np.asarray(p2_3d, dtype=float)[[i1, i2]]

    edge_vec = p2 - p1
    edge_len = np.linalg.norm(edge_vec)
    if edge_len < 1e-8:
        return None
    edge_dir = edge_vec / edge_len

    if perp:
        # Kenara dik yönde, p1 ve p2'den d_L ileri/geri noktalarda kesen çizgiler
        perp_dir = np.array([-edge_dir[1], edge_dir[0]])
        d_L= e/4
        EXT= e/10
        split1 = p1 + edge_dir * d_L
        split2 = p2 - edge_dir * d_L

        # line_L_a = point_unproject_to_3d(polygon, split1)
        # line_L_b = point_unproject_to_3d(polygon, split1 + perp_dir * EXT)
        # line_R_a = point_unproject_to_3d(polygon, split2)
        # line_R_b = point_unproject_to_3d(polygon, split2 + perp_dir * EXT)

        return ((split1, split1 + perp_dir * EXT), (split2, split2 + perp_dir * EXT))

    # w_vector yönünde offset
    w_2d = np.asarray(w_vector, dtype=float)[[i1, i2]]
    w_len = np.linalg.norm(w_2d)
    if w_len < 1e-8:
        return None
    w_dir = w_2d / w_len

    d_L= e/10
    EXT= 500
    
    line_L_p1 = p1 + w_dir * d_L
    line_L_p2 = p2 + w_dir * d_L

    clipped = _clip_line_to_polygon(
        line_L_p1,
        line_L_p2,
        polygon,
        i1,
        i2
    )

    if clipped is None:
        return None

    line_L_p1_ext, line_L_p2_ext = clipped
    return line_L_p1_ext, line_L_p2_ext

    # return (
    #     point_unproject_to_3d(polygon, line_L_p1_ext),
    #     point_unproject_to_3d(polygon, line_L_p2_ext),
    # )

def _split_by_line_2d(poly_2d: np.ndarray, p1: np.ndarray, p2: np.ndarray) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """2D poligonu bir çizgi ile ikiye böler."""
    line_dir = p2 - p1
    if np.linalg.norm(line_dir) < 1e-8:
        return None

    def side(p):
        return line_dir[0] * (p[1] - p1[1]) - line_dir[1] * (p[0] - p1[0])

    pts = poly_2d[:-1] if np.allclose(poly_2d[0], poly_2d[-1]) else poly_2d
    pos, neg = [], []
    n = len(pts)

    for i in range(n):
        A, B = pts[i], pts[(i + 1) % n]
        sa, sb = side(A), side(B)

        if sa >= -1e-8:
            pos.append(A.copy())
        if sa <= 1e-8:
            neg.append(A.copy())

        if sa * sb < -1e-8:
            t = sa / (sa - sb)
            inter = A + t * (B - A)
            pos.append(inter)
            neg.append(inter)

    if len(pos) < 3 or len(neg) < 3:
        return None

    pos = close_polygon(pos)
    neg = close_polygon(neg)

    def area(p):
        return 0.5 * abs(np.sum(p[:-1, 0] * p[1:, 1] - p[1:, 0] * p[:-1, 1]))

    return (pos, neg) if area(pos) >= 1e-4 and area(neg) >= 1e-4 else None

def dict_tree(data, indent=""):
    lines = []

    items = list(data.items())

    for i, (key, value) in enumerate(items):
        last = i == len(items) - 1

        branch = "└── " if last else "├── "
        lines.append(f"{indent}{branch}{key}")

        if isinstance(value, dict):
            new_indent = indent + ("    " if last else "│   ")
            lines.append(dict_tree(value, new_indent))

        else:
            # Yukarıdaki key satırını value ile birleştir
            lines[-1] = f"{indent}{branch}{key} : {value}"

    return "\n".join(lines)


points = {
                "P1": (0.0, 0.0, 0.0),
                "P2": (12000.0, 0.0, 0.0),
                "P3": (12000.0, 9000.0, 0.0),
                "P4": (0.0, 9000.0, 0.0),
                "P5": (0.0, 0.0, 4000.0),
                "P6": (12000.0, 0.0, 4000.0),
                "P7": (12000.0, 9000.0, 4000.0),
                "P8": (0.0, 9000.0, 4000.0),
                "P9": (3000.0, 4000.0, 5000.0),
                "P10": (12000.0, 4000.0, 5000.0),
            }
    
polygons = {
    "D1": ["P1", "P2", "P6", "P5"],
    "D2": ["P1", "P5", "P8", "P4"],
    "D3": ["P2", "P3", "P7", "P10", "P6"],
    "D4": ["P3", "P4", "P8", "P7"],
    "C1": ["P5", "P6", "P10", "P9"],
    "C2": ["P8", "P9", "P10", "P7"],
    "C3": ["P8", "P5", "P9"],
}

building= BuildingWindEngine(points=points, polygons=polygons, scale_factor=1000)

    
            

print("\nbuilding analysis:\n")

wind_parameters= building.get_summary()
print("\nwind_parameters:\n", dict_tree(wind_parameters))

all_surfaces_w= {}
w_list= {"x+":[1,0,0], "y+":[0,1,0], "x-":[-1,0,0], "y-": [0,-1,0]}
for w in w_list:

    all_surfaces_w[w]= building.analysis_all_roof_wind(w_list[w])


def get_same_axis_edge(edges_, leading_edges):
    ed1_same_axis_edges_ = {}
    for ke1_, ed1_ in edges_.items():
        if not ke1_ in leading_edges:
            continue
        
        if ed1_.same_axis:
            
                
            temp_same= []
            for ed1_s in ed1_.same_axis:
                
                
                ed1_surface = ed1_s.get('surface', False)
                ed1_surface_edge = ed1_s.get('surface_edge', False)
                ed1_same_edge = ed1_s.get('same_edge', False)
                if not ed1_same_edge:
                    temp_same.append([ed1_surface, ed1_surface_edge])
            if temp_same:
                ed1_same_axis_edges_[ke1_]= temp_same
                
    return ed1_same_axis_edges_
        
   

def get_surface_results(plane_name, w):
    print("\n>",plane_name)
    print(f"\nWind vector: {w} : {w_list[w]}")
    plane1= all_surfaces_w[w][plane_name]
    polygon= plane1.pts_3d
    
    for kk in ['surface_type', 'polygon', 'angle', 'pitch',  'wind_vector', 'polygon_direction_xy', 'wind_relation', 'global_leading', 'any_shared']:
        print(f" > {kk:20}: {plane1.properties.get(kk, '')}")
    if plane1.surface_type == SurfaceType.WALL:
        return

    exposed_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.exposed]
    leading_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.leading]
    print(f" > exposed_edges: {exposed_edges}")
    print(f" > leading_edges: {leading_edges}")

    same_axis= get_same_axis_edge(plane1.edges, leading_edges)
    
    print(f" > same_axis: {same_axis}")
    # for ke_, ed_ in plane1.edges.items():
    #     print("\n > edge:",ke_)
    #     print("   > points:",ed_.p1, ed_.p2)
    #     print("   > vector_xy:",ed_.vector_xy, "| direction_xy:",ed_.direction_xy)
    #     print("   > angle:",ed_.angle, " | length:",ed_.length)
    #     print("   > vertical:",ed_.vertical)
    #     print("   > exposed:",ed_.exposed)
    #     print("   > leading:",ed_.leading)
    #     print("   > shared:",ed_.shared)
    #     print("   > same_axis:",ed_.same_axis)
    any_shared_= plane1.properties.get("any_shared", False)
    print(f"edge indexes:{leading_edges} FG bölgesi herzaman iki kenardan e/4 kadar offset edilip F bölgeleri bulunacak kalan orta bölge G")

    render_lines= []
    edges= plane1.edges
    e= building.e
    w_vector = w_list[w]
    pts_2d= plane1.pts_2d
    if plane1.wind_relation.value=="WINDWARD":
        print("WINDWARD")
        
        if any_shared_:
            print("DUOPITCH", "HIPPED")
        else:
            print("MONOPITCH")
            
        print("FGH", leading_edges, "kenarlar e/10 offset et ve FG olarak işaretle. poligonun kalan kısmı H")
        print("Bakılacak tablo:",0)

        print("Zone geometry:", polygon)
        for idx_ in leading_edges:
            edge= edges[idx_]
            p1_3d, p2_3d = edge.p1, edge.p2
            res_wind = offset_edge(polygon, p1_3d, p2_3d, w_vector, e)
            res_perp = offset_edge(polygon, p1_3d, p2_3d, w_vector, e, perp=True)
            if res_wind is None or res_perp is None:
                raise RuntimeError("Offset hesaplanamadı")
            line_wind_p1, line_wind_p2 = res_wind
            line_perp1, line_perp2 = res_perp
            (perp1_start, perp1_end), (perp2_start, perp2_end) = res_perp

            print(f"res_wind: {res_wind}")
            print(f"res_wind: {res_perp}")
            

            (pos, neg)= _split_by_line_2d(pts_2d, line_wind_p1, line_wind_p2)

            print("_split_by_line_2d:")
            poly_extract= polygon_edges_to_render(plane1._unproject_2d_to_3d(neg))
            render_lines.append(poly_extract)
            poly_extract= polygon_edges_to_render(plane1._unproject_2d_to_3d(pos))
            render_lines.append(poly_extract)
            
            

            (pos_F1, neg_GF)= _split_by_line_2d(neg, perp1_start, perp1_end)
            (pos_F2, neg_G)= _split_by_line_2d(neg_GF, perp2_start, perp2_end)

            print("_split_by_line_2d:")
            print(" <--:",plane1._unproject_2d_to_3d(neg))
            print(" -->:",plane1._unproject_2d_to_3d(pos))

            print("_split_by_line_2d:")
            print(" <--:F1",plane1._unproject_2d_to_3d(pos_F1))
            print(" -->F2:",plane1._unproject_2d_to_3d(pos_F2))
            print(" -->G:",plane1._unproject_2d_to_3d(neg_G))

            poly_extract= polygon_edges_to_render(plane1._unproject_2d_to_3d(pos_F1))
            render_lines.append(poly_extract)
            poly_extract= polygon_edges_to_render(plane1._unproject_2d_to_3d(pos_F2))
            render_lines.append(poly_extract)
            poly_extract= polygon_edges_to_render(plane1._unproject_2d_to_3d(neg_G))
            render_lines.append(poly_extract)

            
        

    elif plane1.wind_relation.value=="LEEWARD":
        print("LEEWARD")
        if any_shared_:
            print("DUOPITCH", "HIPPED")
            print("DUOPITCH veya HIPPED çatı tipi farketmiyor ama cpe değerlerini kontrol et")
            print("DUOPITCH için JI HIPPED için KJI.. önce türbülans alanı için leading_edges kenarları e/10 offset yap",  f"edge indexes:{leading_edges}", "K, diğer exposed kenarlar:",f"edge indexes:{exposed_edges}","J olur. yüzeyin geri kalanı her zaman I")
        else:
            print("MONOPITCH")
            print("Bakılacak tablo:",180)
            print("FGH", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle. alçak saçak tarafı Fl, yüksek saçak tarafı Fu poligonun kalan kısmı H")
        
    elif plane1.wind_relation.value=="PARALLEL":
        print("PARALLEL")
        if any_shared_:
            
            if same_axis:
                print("DUOPITCH")
                print("Bakılacak tablo:",90)
                print("FGHI",  f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle.")
                print("FG Bölgesi aynı akstaki diğer kenara uzayacak.", f"edge indexes:{leading_edges} alçak kot e/4 mesafede F kalan bölge G")
                print("poligonun kalan kısmı e/2 ye kadar H sonrası I")
            else:
                print("HIPPED")
                print("Bakılacak tablo:",0)
                print("LMN", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve L olarak işaretle. poligonun kalan kısmı e/2 ye kadar M sonrası N")
        else:
            print("MONOPITCH")
            print("Bakılacak tablo:",90)
            print("FGH", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle. poligonun kalan kısmı e/2 ye kadar H sonrası I")

    return render_lines
w= "y+"
render_lines= get_surface_results("C1", w)
from canvas3d import View3D

import sys
from PySide6.QtWidgets import QApplication

def get_app():
    app = QApplication.instance()
    if app is None:
        app = QApplication([])
    return app


app = get_app()

view = View3D()
view.set_data(
    points={"O": [0, 0, 0]},
    # lines=render_lines
)

view.resize(1000, 700)
view.show()

In [ ]:
render_lines

In [ ]:
plane_name= "C1"
w= "y+"
print("\n>",plane_name)
print(f"\nWind vector: {w} : {w_list[w]}")
plane1= all_surfaces_w[w][plane_name]
plane1_edges_ = polygon_edges(plane1.pts_3d)
render_lines= [[edg['p1'], edg['p2']] for edg in plane1_edges_]
 
from canvas3d import View3D

import sys
from PySide6.QtWidgets import QApplication

app = QApplication(sys.argv)







view = View3D()
view.set_data(points= {"O": [0,0,0]}, lines= render_lines)

view.resize(1000, 700)
view.show()
view.zoom_extents()

sys.exit(app.exec())



In [ ]:
def get_same_axis_edge(edges_):
    ed1_same_axis_edges_ = {}
    for ke1_, ed1_ in edges_.items():
        
        ed1_same_axis= ed1_.same_axis
        if ed1_same_axis:
            temp_same= []
            for ed1_s in ed1_.same_axis:
                ed1_surface = ed1_s.get('surface', False)
                ed1_surface_edge = ed1_s.get('surface_edge', False)
                ed1_same_edge = ed1_s.get('same_edge', False)
                if not ed1_same_edge:
                    temp_same.append([ed1_surface, ed1_surface_edge])
            if temp_same:
                ed1_same_axis_edges_[ke1_]= temp_same
                
    return ed1_same_axis_edges_
        
   

def get_surface_results(plane_name, w):
    print("\n>",plane_name)
    print(f"\nWind vector: {w} : {w_list[w]}")
    plane1= all_surfaces_w[w][plane_name]
    for kk in ['surface_type', 'polygon', 'angle', 'pitch',  'wind_vector', 'polygon_direction_xy', 'wind_relation', 'global_leading', 'any_shared']:
        print(f" > {kk:20}: {plane1.properties.get(kk, '')}")
    if plane1.surface_type == SurfaceType.WALL:
        return

    exposed_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.exposed]
    leading_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.leading]
    print(f" > exposed_edges: {exposed_edges}")
    print(f" > leading_edges: {leading_edges}")

    same_axis= get_same_axis_edge(plane1.edges)
    
    print(f" > same_axis: {same_axis}")
    # for ke_, ed_ in plane1.edges.items():
    #     print("\n > edge:",ke_)
    #     print("   > points:",ed_.p1, ed_.p2)
    #     print("   > vector_xy:",ed_.vector_xy, "| direction_xy:",ed_.direction_xy)
    #     print("   > angle:",ed_.angle, " | length:",ed_.length)
    #     print("   > vertical:",ed_.vertical)
    #     print("   > exposed:",ed_.exposed)
    #     print("   > leading:",ed_.leading)
    #     print("   > shared:",ed_.shared)
    #     print("   > same_axis:",ed_.same_axis)
    any_shared_= plane1.properties.get("any_shared", False)
    print(f"edge indexes:{leading_edges} FG bölgesi herzaman iki kenardan e/4 kadar offset edilip F bölgeleri bulunacak kalan orta bölge G")
    if plane1.wind_relation.value=="WINDWARD":
        print("WINDWARD")
        
        if any_shared_:
            print("DUOPITCH", "HIPPED")
        else:
            print("MONOPITCH")
            
        print("FGH", leading_edges, "kenarlar e/10 offset et ve FG olarak işaretle. poligonun kalan kısmı H")
        print("Bakılacak tablo:",0)

    elif plane1.wind_relation.value=="LEEWARD":
        print("LEEWARD")
        if any_shared_:
            print("DUOPITCH", "HIPPED")
            print("DUOPITCH veya HIPPED çatı tipi farketmiyor ama cpe değerlerini kontrol et")
            print("DUOPITCH için JI HIPPED için KJI.. önce türbülans alanı için leading_edges kenarları e/10 offset yap",  f"edge indexes:{leading_edges}", "K, diğer exposed kenarlar:",f"edge indexes:{exposed_edges}","J olur. yüzeyin geri kalanı her zaman I")
        else:
            print("MONOPITCH")
            print("Bakılacak tablo:",180)
            print("FGH", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle. alçak saçak tarafı Fl, yüksek saçak tarafı Fu poligonun kalan kısmı H")
        
    elif plane1.wind_relation.value=="PARALLEL":
        print("PARALLEL")
        if any_shared_:
            
            if same_axis:
                print("DUOPITCH")
                print("Bakılacak tablo:",90)
                print("FGHI",  f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle.")
                print("FG Bölgesi aynı akstaki diğer kenara uzayacak.", f"edge indexes:{leading_edges} alçak kot e/4 mesafede F kalan bölge G")
                print("poligonun kalan kısmı e/2 ye kadar H sonrası I")
            else:
                print("HIPPED")
                print("Bakılacak tablo:",0)
                print("LMN", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve L olarak işaretle. poligonun kalan kısmı e/2 ye kadar M sonrası N")
        else:
            print("MONOPITCH")
            print("Bakılacak tablo:",90)
            print("FGH", f"edge indexes:{leading_edges}", "kenarlar e/10 offset et ve FG olarak işaretle. poligonun kalan kısmı e/2 ye kadar H sonrası I")
        
for w in w_list:
    get_surface_results("C1", w)

In [ ]:
for plane_name in polygons:

    
    print("\n>",plane_name)
    for w in w_list:
        print(f"\nWind vector: {w} : {w_list[w]}")
        plane1= all_surfaces_w[w][plane_name]
        for kk in ['surface_type', 'polygon', 'angle', 'pitch',  'wind_vector', 'polygon_direction_xy', 'wind_relation', 'global_leading', 'any_shared']:
            print(f" > {kk:20}: {plane1.properties.get(kk, '')}")
        if plane1.surface_type == SurfaceType.WALL:
            continue

        exposed_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.exposed]
        leading_edges= [ke_ for ke_, ed_ in plane1.edges.items() if ed_.leading]
        print(f" > exposed_edges: {exposed_edges}")
        print(f" > leading_edges: {leading_edges}")
        
        for ke_, ed_ in plane1.edges.items():
            print("\n > edge:",ke_)
            print("   > points:",ed_.p1, ed_.p2)
            print("   > vector_xy:",ed_.vector_xy, "| direction_xy:",ed_.direction_xy)
            print("   > angle:",ed_.angle, " | length:",ed_.length)
            print("   > vertical:",ed_.vertical)
            print("   > exposed:",ed_.exposed)
            print("   > leading:",ed_.leading)
            print("   > shared:",ed_.shared)
            print("   > same_axis:",ed_.same_axis)